# CRIVO — Pensador v2: avaliar o que já foi salvo (quando a sessão caiu antes do fim)

O treino (`pensador_v2_colab.ipynb`) guarda no Drive a melhor versão a cada medida (`pensador.pt`). Se a sessão caiu antes da avaliação final, este notebook **não treina de novo**: pega a pasta `v2_*` mais nova que tem `pensador.pt` e roda a mesma avaliação final, com os mesmos conjuntos congelados e os mesmos gates registrados antes.

> Na **mesma conta** do treino (G). GPU (T4 serve) → **Executar tudo** → mande o print da última célula.

In [ ]:
# Célula 0 — GPU, Drive e a pasta do treino
import os, sys, json, subprocess, hashlib, shutil
from pathlib import Path
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sem GPU (vai na CPU, mais devagar)", flush=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
RAIZ = Path("/content/drive/MyDrive/CRIVO/pensador-20261012")
pastas = sorted(p for p in RAIZ.glob("v2_*") if (p / "v2" / "pensador.pt").is_file())
if not pastas:
    raise SystemExit("ERRO: não achei nenhum pensador.pt em CRIVO/pensador-20261012/v2_*/v2. Rode na conta do treino.")
SAIDA = pastas[-1]
ck = torch.load(SAIDA / "v2" / "pensador.pt", map_location="cpu")
print("pasta:", SAIDA, "| melhor passo salvo:", ck["passo"], "| já avaliado antes:", (SAIDA / "v2" / "resultado_pensador_v2.json").is_file(), flush=True)

In [ ]:
# Célula 1 — código e conjuntos congelados (os mesmos do treino)
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
EXP = REPO / "experimentos" / "pensador_20261012"
ESPERADO = "20deb41adcb704e791027ba5939feb253f6cbd59943de633389eee841500c3df"
assert hashlib.sha256((EXP / "conjuntos_pensador.json").read_bytes()).hexdigest() == ESPERADO, "conjuntos diferentes"
cfg = ck["config"]
cmd = [sys.executable, "-u", str(EXP / "treinar_pensador_v2.py"), "--saida", str(SAIDA / "v2"), "--so-avaliar",
       "--d", str(cfg["d"]), "--camadas", str(cfg["camadas"]), "--voltas", str(cfg["voltas"])]
with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for linha in p.stdout:
        print(linha, end="", flush=True)
if p.returncode:
    raise SystemExit("avaliação falhou (código %d)" % p.returncode)
v2 = json.loads((SAIDA / "v2" / "resultado_pensador_v2.json").read_text())

In [ ]:
# Célula 3 — resumo: as três contas (a aprovação usa "adaptativo_busca", registrado antes)
V1 = {"blind_iid": 0.845, "ood_profundidade": 0.579, "ood_tamanho": 0.646}
print("parâmetros", v2["parametros"], "| passo escolhido", v2["passo_escolhido"])
for nome in ("blind_iid", "ood_profundidade", "ood_tamanho"):
    print("\n" + nome, "(v1: exato %.1f%%)" % (100 * V1[nome]))
    for conta in ("rede", "adaptativo", "adaptativo_busca"):
        m = v2["avaliacao"][nome][conta]
        print("  %-17s exato %5.1f%%  status %5.1f%%  erradas(rede) %4.1f%%  cobertura segura %5.1f%%  voltas %.1f" % (
            conta, 100 * m["exato"], 100 * m["status_acc"], 100 * m["afirmacoes_erradas_brutas"],
            100 * m["politica_cobertura"], m["voltas_media"]))
print("\ninvariância", round(100 * v2["invariancia_status"], 1), "% | gates", v2["gates"])
print("APROVADO:", v2["aprovado"], "| ativo_no_chat:", v2["ativo_no_chat"])
print("Arquivos em", SAIDA, "— avise o assistente.")